In [ ]:
!pip install transformers datasets torch scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import torch


In [ ]:
dataset = load_dataset("imdb")

print(dataset)
print(dataset['train'][0])


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])


In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir='./logs',
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()


In [ ]:
pred_output = trainer.predict(dataset["test"])
preds = np.argmax(pred_output.predictions, axis=1)
labels = pred_output.label_ids

print(classification_report(labels, preds))


In [ ]:
cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, cmap='Blues', fmt='g')
plt.title("Confusion Matrix - BERT Sentiment Classifier")
plt.show()


In [ ]:
probs = torch.softmax(torch.tensor(pred_output.predictions), dim=1)[:,1]
fpr, tpr, _ = roc_curve(labels, probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.title("ROC Curve (AUC = %.2f)" % roc_auc)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()


In [ ]:
trainer.save_model("./models/bert-sentiment")
tokenizer.save_pretrained("./models/bert-sentiment")


In [ ]:
text = "This movie was amazing and emotional!"

tokens = tokenizer(text, return_tensors="pt", truncation=True)
output = model(**tokens)
prediction = torch.argmax(output.logits).item()

print("Sentiment:", "Positive" if prediction == 1 else "Negative")